In [ ]:
import os
import sys

ENDWITHS = 'OpenMantra'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..', '..'))
print(BASE_DIR)

sys.path.insert(0, os.path.join(BASE_DIR, 'src'))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
from OpenMantraEvaluator import OpenMantraEvaluator
from pipeline.TranslationModels.ElanMtJaEnBatchTranslator import ElanMtJaEnBatchTranslator

from typing import List

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.abspath(os.path.join(BASE_DIR, '..', '.env')))

HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
OPENMANTRA_ROOT = os.path.join(BASE_DIR, 'data', 'open-mantra-dataset')
print(f"OpenMantra dataset root: {OPENMANTRA_ROOT}")

SAVE_DIR = os.path.join(BASE_DIR, 'output', 'pipeline', 'Evaluators', 'Translators', 'OpenMantra')

In [ ]:
evaluator = OpenMantraEvaluator(
    openmantra_root=OPENMANTRA_ROOT,
    source_lang='text_ja',
    target_lang='text_en'
)

In [ ]:
translation_model = ElanMtJaEnBatchTranslator()

In [ ]:
# Preprocess
def strip_text(xs): return [x.strip() for x in xs]
def normalize_punct(xs): return [x.replace("。", ".") for x in xs]

# Postprocess
# def fix_spacing(xs): return [x.replace(" ,", ",") for x in xs]
def truncate_long_predictions(preds: List[str], source_texts: List[str], factor: int = 2) -> List[str]:
    # truncate only when the model clearly over-generated
    if len(preds) > factor * len(source_texts):
        return preds[: len(source_texts)]
    return preds  # must return the unchanged list otherwise

# Add preprocess
translation_model.add_preprocess_step("strip", strip_text)
translation_model.add_preprocess_step("normalize_punct", normalize_punct)

# Add postprocess
# translation_model.add_postprocess_step("fix_spacing", fix_spacing)
translation_model.add_postprocess_step("truncate_long_predictions", truncate_long_predictions)

translation_model.load_model()

In [ ]:
# Run evaluation
metrics = evaluator.evaluate(translation_model, save_dir=SAVE_DIR)